[Version C#](Z3-10-Cryptarithmetic-CSharp.ipynb) | ← [09 - Enigme d'Einstein](Z3-09-Enigme-Einstein-Python.ipynb) | [README Z3-Python](README.md) | [11 - Coloration de graphe >>](Z3-11-Graph-Coloring-Python.ipynb)
# 10. Cryptarithmes (SEND + MORE = MONEY)
Un **cryptarithme** est une equation arithmetique ou chaque lettre represente un **chiffre distinct** (0-9), et ou les mots formant l'equation correspondent a des nombres en ecriture positionnelle. Le classique `SEND + MORE = MONEY` admet une unique solution : laquelle ?
Ce type de puzzle est un cas canonique de **satisfaction de contraintes sur les entiers** : 8 lettres, chacune dans `{0..9}`, deux a deux distinctes, avec des tetes non nulles (S et M) et une equation lineaire positionnelle. La **propagation de contraintes** du solveur SMT resout instantanement ce que la brute force met plusieurs centaines de millisecondes a enumerer.
> Port pyz3 du notebook C# [`13_Cryptarithmetic_SMT.ipynb`](../Z3-Linq2Z3/13_Cryptarithmetic_SMT.ipynb) (Microsoft.Z3 brut).

In [1]:
from z3 import *
import time
print("Imports OK : z3-solver (pyz3)")


Imports OK : z3-solver (pyz3)


## Modelisation : SEND + MORE = MONEY

On encode chaque lettre comme un entier `Int` dans `{0..9}`, on impose la **distinction globale** (`Distinct`), les têtes non nulles (`S > 0`, `M > 0`), puis l'**equation positionnelle** :

```
  1000*S + 100*E + 10*N + D     (SEND)
+ 1000*M + 100*O + 10*R + E     (MORE)
= 10000*M + 1000*O + 100*N + 10*E + Y   (MONEY)
```

Z3 propage ces contraintes et renvoie l'unique modèle.

In [2]:
# Cryptarithme classique : SEND + MORE = MONEY
s = Solver()
S, E, N, D, M, O, R, Y = Ints('S E N D M O R Y')
letters = [S, E, N, D, M, O, R, Y]

# (1) Domaine 0..9 pour chaque lettre
s.add([And(v >= 0, v <= 9) for v in letters])

# (2) Les 8 lettres sont deux a deux distinctes (contrainte globale)
s.add(Distinct(letters))

# (3) Tetes non nulles (pas de zero non-significatif)
s.add(S > 0, M > 0)

# (4) Equation arithmetique positionnelle : SEND + MORE == MONEY
s.add((1000*S + 100*E + 10*N + D) +
      (1000*M + 100*O + 10*R + E) ==
      (10000*M + 1000*O + 100*N + 10*E + Y))

if s.check() != sat:
    print("UNSAT : aucune solution (inattendu pour ce cryptarithme)")
else:
    m = s.model()
    send  = 1000*m[S].as_long() + 100*m[E].as_long() + 10*m[N].as_long() + m[D].as_long()
    more  = 1000*m[M].as_long() + 100*m[O].as_long() + 10*m[R].as_long() + m[E].as_long()
    money = 10000*m[M].as_long() + 1000*m[O].as_long() + 100*m[N].as_long() + 10*m[E].as_long() + m[Y].as_long()
    print("  S=%d E=%d N=%d D=%d  M=%d O=%d R=%d Y=%d" % (m[S].as_long(), m[E].as_long(), m[N].as_long(), m[D].as_long(),
                                                            m[M].as_long(), m[O].as_long(), m[R].as_long(), m[Y].as_long()))
    print("    %5d" % send)
    print("  + %5d" % more)
    print("  -------")
    print("  %5d    (%d + %d = %d, verifie = %s)" % (money, send, more, money, send + more == money))


  S=9 E=5 N=6 D=7  M=1 O=0 R=8 Y=2
     9567
  +  1085
  -------
  10652    (9567 + 1085 = 10652, verifie = True)


## Interpretation

La solution unique est `S=9 E=5 N=6 D=7 M=1 O=0 R=8 Y=2`, soit `9567 + 1085 = 10652`. Notez le rôle de la **retenue** : ce n'est pas evident a l'oeil, c'est la propagation arithmetique du solveur qui la deduit automatiquement a partir de l'equation positionnelle. Aucune boucle explicite sur les retenues n'est necessaire : c'est précisément ce que Z3 fait mieux qu'un code imperatif.

## Anatomie : où sont passées les retenues ?

L'encodage ci-dessus n'introduit **aucune variable de retenue** — et pourtant l'addition posée à la main en exige. Reconstituons la chaîne des retenues que la propagation de Z3 retrouve implicitement, en notant $c_i$ la retenue sortant de la colonne $i$ (de droite à gauche) :

| Colonne | Équation manuelle | Avec la solution mesurée |
|---|---|---|
| unités | $D + E = 10 c_1 + Y$ | $7 + 5 = 12$, donc $c_1 = 1$, $Y = 2$ |
| dizaines | $N + R + c_1 = 10 c_2 + E$ | $6 + 8 + 1 = 15$, donc $c_2 = 1$, $E = 5$ |
| centaines | $E + O + c_2 = 10 c_3 + N$ | $5 + 0 + 1 = 6$, donc $c_3 = 0$, $N = 6$ |
| milliers | $S + M + c_3 = 10 c_4 + O$ | $9 + 1 + 0 = 10$, donc $c_4 = 1$, $O = 0$ |
| dizaines de milliers | $c_4 = M$ | $1 = M$ |

Toute cette chaîne est **subsumée** par l'unique équation linéaire positionnelle : l'arithmétique entière de Z3 raisonne sur l'égalité $9567 + 1085 = 10652$ comme sur n'importe quelle identité entière, sans qu'on ait à décomposer les colonnes. Le chiffre de tête $M = 1$ n'est rien d'autre que la dernière retenue devenue chiffre significatif — le solveur la déduit, il ne la déclare pas.

## Approche naive vs solveur : que fait la propagation ?

La brute force enumere tous les **arrangements** `P(10, 8) = 10!/(10-8)! = 1 814 400` candidats (8 lettres distinctes prises parmi 10 chiffres), puis teste l'equation pour chacun. Le solveur, lui, **propage** les contraintes (distinction, domaines, equation) pour eliminer en bloc des familles entieres d'affectations impossibles avant même de les tester. Mesurons l'ecart.

In [3]:
# (1) Brute force : 8 boucles imbriquees, on saute des que deux lettres coincident
#     P(10,8) = 1 814 400 candidats distincts testes.
nb_brute = 0
t0 = time.perf_counter()
for S in range(10):
 for E in range(10):
  if E == S: continue
  for N in range(10):
   if N in (S, E): continue
   for D in range(10):
    if D in (S, E, N): continue
    for M in range(10):
     if M in (S, E, N, D): continue
     for O in range(10):
      if O in (S, E, N, D, M): continue
      for R in range(10):
       if R in (S, E, N, D, M, O): continue
       for Y in range(10):
        if Y in (S, E, N, D, M, O, R): continue
        if S > 0 and M > 0:
            send  = 1000*S + 100*E + 10*N + D
            more  = 1000*M + 100*O + 10*R + E
            money = 10000*M + 1000*O + 100*N + 10*E + Y
            if send + more == money:
                nb_brute += 1
t_brute = time.perf_counter() - t0

# (2) Z3 : meme theoreme que la cellule precedente, mesure
t0 = time.perf_counter()
nb_z3 = 0
s2 = Solver()
S2, E2, N2, D2, M2, O2, R2, Y2 = Ints('S2 E2 N2 D2 M2 O2 R2 Y2')
L2 = [S2, E2, N2, D2, M2, O2, R2, Y2]
s2.add([And(v >= 0, v <= 9) for v in L2])
s2.add(Distinct(L2))
s2.add(S2 > 0, M2 > 0)
s2.add((1000*S2 + 100*E2 + 10*N2 + D2) + (1000*M2 + 100*O2 + 10*R2 + E2) ==
       (10000*M2 + 1000*O2 + 100*N2 + 10*E2 + Y2))
if s2.check() == sat:
    nb_z3 = 1
t_z3 = time.perf_counter() - t0

print("Brute force : %d solution(s) en %.1f ms  (1 814 400 candidats testes)" % (nb_brute, t_brute * 1000))
print("Z3 solve    : %d solution(s) en %.1f ms  (propagation de contraintes)" % (nb_z3, t_z3 * 1000))
print(" -> meme resultat (%s), strategies differentes." % (nb_brute == nb_z3))


Brute force : 1 solution(s) en 6611.1 ms  (1 814 400 candidats testes)
Z3 solve    : 1 solution(s) en 60.6 ms  (propagation de contraintes)
 -> meme resultat (True), strategies differentes.


## Lecture du benchmark

L'ecart est net : la brute force teste **1 814 400** affectations pour trouver l'unique solution, tandis que Z3 la trouve en quelques millisecondes via propagation. Ce n'est pas une question de vitesse brute du langage : c'est la **nature de la recherche**. Le solveur ne << teste >> pas, il **deduit** - les contraintes `Distinct` et l'equation positionnelle reduisent l'espace avant tout branchement. C'est l'intérêt pedagogique central : sur un problème combinatoire, declarer les contraintes bat l'enumeration imperative.

## Témoin et preuve : deux niveaux de garantie

`check()` renvoie `sat` et un modèle : c'est un **témoin** de l'existence d'au moins une solution, pas une preuve d'unicité. L'exemple guidé 2 montre comment bloquer les modèles trouvés et chercher à établir leur exhaustivité. Une réponse `unknown` ne prouve pas l'absence d'autres modèles.

## Variante : CROSS + ROADS = DANGER (9 lettres distinctes)

On complique : 9 lettres distinctes (`C, R, O, S, A, D, N, G, E`), trois têtes non nulles (`C`, `R`, `D`), et une equation a 6 chiffres en résultats. La brute force devrait enumerer `P(10, 9) = 3 628 800` candidats - le solveur reste instantane.

In [4]:
# Variante : CROSS + ROADS = DANGER (9 lettres distinctes)
s3 = Solver()
C, R, O, S, A, D, N, G, E = Ints('C R O S A D N G E')
L3 = [C, R, O, S, A, D, N, G, E]

# Domaine 0..9 pour les 9 lettres
s3.add([And(v >= 0, v <= 9) for v in L3])

# 9 lettres deux a deux distinctes
s3.add(Distinct(L3))

# Tetes non nulles : C (CROSS), R (ROADS), D (DANGER)
s3.add(C > 0, R > 0, D > 0)

# CROSS + ROADS == DANGER
# CROSS  = 10000*C + 1000*R + 100*O + 10*S + S
# ROADS  = 10000*R + 1000*O + 100*A + 10*D + S
# DANGER = 100000*D + 10000*A + 1000*N + 100*G + 10*E + R
s3.add((10000*C + 1000*R + 100*O + 10*S + S) +
       (10000*R + 1000*O + 100*A + 10*D + S) ==
       (100000*D + 10000*A + 1000*N + 100*G + 10*E + R))

if s3.check() != sat:
    print("UNSAT : aucune solution pour CROSS + ROADS = DANGER")
else:
    m = s3.model()
    cross  = 10000*m[C].as_long() + 1000*m[R].as_long() + 100*m[O].as_long() + 10*m[S].as_long() + m[S].as_long()
    roads  = 10000*m[R].as_long() + 1000*m[O].as_long() + 100*m[A].as_long() + 10*m[D].as_long() + m[S].as_long()
    danger = 100000*m[D].as_long() + 10000*m[A].as_long() + 1000*m[N].as_long() + 100*m[G].as_long() + 10*m[E].as_long() + m[R].as_long()
    print("CROSS=%d  ROADS=%d  DANGER=%d   (%d + %d = %d, egal a DANGER = %d : %s)" %
          (cross, roads, danger, cross, roads, cross + roads, danger, cross + roads == danger))


CROSS=96233  ROADS=62513  DANGER=158746   (96233 + 62513 = 158746, egal a DANGER = 158746 : True)


## Lecture de la variante CROSS + ROADS = DANGER

La sortie présente un modèle : vérifiez que les deux opérandes donnent bien le résultat affiché. La somme de deux nombres à cinq chiffres reste inférieure à 200 000 : la tête `D` du résultat à six chiffres vaut donc nécessairement 1. La lettre `S` apparaît deux fois dans CROSS et une fois dans ROADS : ces occurrences partagent une seule variable. Neuf lettres distinctes occupent neuf chiffres ; le modèle affiché ne suffit pas à établir son unicité.

## Exemples guidés

Les trois premières réalisations, de Léane Audegond (EPF) et issues du deuxième TP, illustrent la modélisation d'une addition, l'énumération des modèles et une multiplication non linéaire.

Les quatre réalisations suivantes, de @Will-emma et issues de la PR #28098, prolongent ces méthodes : retenues explicites et équivalence d'encodages, bases variables, noyau insatisfiable et multiplication par constante. Les exercices non résolus viennent ensuite.

### Exemple guidé 1 — DONALD + GERALD = ROBERT

Réalisation de Léane Audegond (EPF), issue du deuxième TP.

On modélise dix lettres distinctes, les chiffres de tête non nuls et l'égalité positionnelle :

- `DONALD = 100000*D + 10000*O + 1000*N + 100*A + 10*L + D`
- `GERALD = 100000*G + 10000*E + 1000*R + 100*A + 10*L + D`
- `ROBERT = 100000*R + 10000*O + 1000*B + 100*E + 10*R + T`

Dans DONALD, D apparaît en tête et aux unités ; dans ROBERT, R apparaît en tête et aux dizaines.

In [5]:
# Exercice 1 : DONALD + GERALD = ROBERT
# TODO etudiant :
# Etape 1 : D, O, N, A, L, G, E, R, B, T = Ints('D O N A L G E R B T')
# Etape 2 : domaine 0..9 pour les 10 lettres
# Etape 3 : Distinct(...) sur les 10 lettres
# Etape 4 : têtes non nulles (D > 0, G > 0, R > 0)
# Etape 5 : equation DONALD + GERALD == ROBERT
#result = None  # TODO etudiant
#print("Exercice 1 a completer : modelisez DONALD + GERALD = ROBERT avec Z3")


from z3 import *
def exercice_1():
    
    D, O, N, A, L, G, E, R, B, T = Ints('D O N A L G E R B T') #on créé 10 inconnues entières (int)
    lettres = [D, O, N, A, L, G, E, R, B, T] #regorupe les variables dans une seule liste pour ne pas les réécrire plus tard à chaque fois

    s = Solver()   # le solveur qui va recevoir les règles

    # chaque lettre est un chiffre entre 0 et 9
    s.add([And(v >= 0, v <= 9) for v in lettres])

    # deux lettres différentes = deux chiffres différents
    s.add(Distinct(lettres))

    # la première lettre de chaque mot n'est pas 0
    s.add(D > 0, G > 0, R > 0)

    # l'addition posée en colonnes.
    # chaque lettre est multipliée par son rang (unités, dizaines, centaines...)
    donald = 100000*D + 10000*O + 1000*N + 100*A + 10*L + D  
    gerald = 100000*G + 10000*E + 1000*R + 100*A + 10*L + D
    robert = 100000*R + 10000*O + 1000*B + 100*E + 10*R + T
    s.add(donald + gerald == robert)

    # On lance la recherche
    # on va demander s'il y a une solution avec une boucle if 
    #on regarde d'abord s'il y en a une avec s.check() == sat
    if s.check() == sat: 
        m = s.model() #si oui on recupere le model et on l'affiche
        v = {str(x): m[x].as_long() for x in lettres}   # dictionnaire qui associe chaque lettre à son chiffre
        print("Solution :", v)  #affiche la solution trouvée

        # On reconstruit les 3 nombres pour vérifier si c'est correct
        #on va donc chercher dans le dictionnaire les valeurs pour chaque lettre, les transformer en texte et les assembler 
        n_donald = int("".join(str(v[c]) for c in "DONALD"))
        n_gerald = int("".join(str(v[c]) for c in "GERALD"))
        n_robert = int("".join(str(v[c]) for c in "ROBERT"))
        print(f"{n_donald} + {n_gerald} = {n_robert}") #ici on affiche le resultat de l'addition pour verifier que c'est correct 
        
    else: # si aucune solution n'est trouvée, on affiche un message
        print("Aucune solution")

exercice_1()
print()

Solution : {'D': 5, 'O': 2, 'N': 6, 'A': 4, 'L': 8, 'G': 1, 'E': 9, 'R': 7, 'B': 3, 'T': 0}
526485 + 197485 = 723970



### Exemple guidé 2 — Énumérer les solutions de SEND + MORE = MONEY

Réalisation de Léane Audegond (EPF), issue du deuxième TP. Après chaque modèle, une disjonction impose qu'au moins une lettre change. Quand la recherche se termine par `unsat`, tous les modèles ont été énumérés ; un seul modèle établit alors l'unicité.

In [6]:
# Exercice 2 : prouver l'unicite de SEND + MORE = MONEY
# TODO etudiant :
# 1. Re-resoudre, recuperer le modele m
# 2. Ajouter Or(S != m[S], E != m[E], N != m[N], ...) pour nier la solution
# 3. Énumérer par itération ; si le statut terminal est unsat, l'énumération est
#    exhaustive (unicité si 1 solution) ; si Z3 répond unknown, le compte reste
#    partiel : ni exhaustivité ni unicité ne sont prouvées
#nb_solutions = None  # TODO etudiant
#print("Exercice 2 a completer : prouvez l'unicite (ou enumerez toutes les solutions)")


def exercice_2():
    S, E, N, D, M, O, R, Y = Ints('S E N D M O R Y')
    lettres = [S, E, N, D, M, O, R, Y]

    s = Solver()
    s.add([And(v >= 0, v <= 9) for v in lettres])    # chiffres 0..9
    s.add(Distinct(lettres))                         # tous différents
    s.add(S > 0, M > 0)                              # 1eres lettres non nulles
    s.add((1000*S + 100*E + 10*N + D) +
          (1000*M + 100*O + 10*R + E) ==
          (10000*M + 1000*O + 100*N + 10*E + Y))     # l'addition


    nb_solutions = 0 #initie le compteur de solutions

    # Boucle while tant que Z3 trouve une solution, on la note,
    # puis on lui interdit cette solution et on redemande.
    # On garde le statut du dernier check() : la boucle peut aussi sortir
    # sur unknown (limite atteinte, incomplétude), pas seulement sur unsat

    terminal = sat
    while terminal == sat:
        terminal = s.check()
        if terminal != sat:
            break

        m = s.model() #affiche le model de la solution trouvée

        nb_solutions += 1 #fait monter le compteur de solutions à chaque nouvelle solutions trouvée

        #on affiche le numéro de la solution et le dictionnaire qui associe chaque lettre à son chiffre
        print("Solution", nb_solutions, ":", {str(v): m[v].as_long() for v in lettres}) 

        # on va ajouter une nouvelle contrainte pour interdire la solution que l'on vient de trouver
        # et forcer Z3 à chercher une autre solution
        # Test si au moins une lettre est différente de la solution trouvée avant
        s.add(Or([v != m[v] for v in lettres]))

    # Seul un statut terminal unsat prouve l'exhaustivité : unknown veut dire
    # que Z3 n'a pas conclu -- le compte n'énumère que les solutions trouvées,
    # ni l'exhaustivité ni l'unicité ne sont prouvées
    if terminal == unsat:
        print("Nombre total de solutions :", nb_solutions)

        if nb_solutions == 1: #affiche le nombre de solutions trouvées, et si c'est 1, on affiche que la solution est unique
            print("-> La solution est unique")
    else:
        print("unknown : Z3 n'a pas conclu --", nb_solutions,
              "solution(s) trouvée(s), énumération incomplète (ni exhaustivité ni unicité prouvées)")
    return nb_solutions

exercice_2()
print()

Solution 1 : {'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O': 0, 'R': 8, 'Y': 2}
Nombre total de solutions : 1
-> La solution est unique



### Exemple guidé 3 — Une multiplication sans solution

Réalisation de Léane Audegond (EPF), issue du deuxième TP. On cherche `ABC * D = DBC` en base dix, avec quatre chiffres distincts et `A, D > 0`. Contrairement à une multiplication par une constante, le produit par `D` est non linéaire. Le résultat `unsat` concerne précisément ces contraintes ; il ne démontre pas l'impossibilité d'autres multiplications.

In [7]:
# Exercice 3 : ABC * D = DBC (multiplication)
# TODO etudiant :
# 1. A, B, C, D = Ints('A B C D'), domaine 0..9
# 2. Distinct(A, B, C, D), têtes A > 0 et D > 0
# 3. Equation : (100*A + 10*B + C) * D == (100*D + 10*B + C)
# 4. Distinguer les trois statuts : sat (solution), unsat (absence prouvée),
#    unknown (Z3 n'a pas conclu -- ni l'un ni l'autre)
#result = None  # TODO etudiant
#print("Exercice 3 a completer : modelisez ABC * D = DBC")

def exercice_3():
    A, B, C, D = Ints('A B C D')   # on crée les 4 variables
    lettres = [A, B, C, D]         # on les range dans une liste

    s = Solver()
    s.add([And(v >= 0, v <= 9) for v in lettres])   # chiffres entre 0 et 9
    s.add(Distinct(lettres)) # 4 chiffres tous différents
    s.add(A > 0, D > 0) # 1ere lettres non nulles
    
    s.add((100*A + 10*B + C) * D == (100*D + 10*B + C))   # ABC * D = DBC

    verdict = s.check() #statut terminal explicite
    if verdict == sat: #une solution est trouvée
        m = s.model() #affiche le model de la solution trouvée
        print("Solution :", {str(v): m[v].as_long() for v in lettres}) #print le dictionnaire qui associe chaque lettre à son chiffre
    elif verdict == unsat:
        # unsat : l'équation force A = 1, puis soit D = 1, soit B = C = 0,
        # ce qui est impossible si les 4 chiffres doivent être différents
        print("unsat : aucune solution avec 4 chiffres distincts") #si aucune solution n'est trouvée, on affiche un message
    else:
        # unknown : Z3 n'a pas conclu -- ni une solution, ni son absence, ne sont prouvées
        print("unknown : Z3 n'a pas conclu -- ni solution ni absence de solution prouvée")


exercice_3()
print()

unsat : aucune solution avec 4 chiffres distincts



### Exemple guidé 4 — Retenues et équivalence

Réalisation de @Will-emma, issue de la PR #28098.

`SEND + MORE = MONEY` est encodé deux fois : une seule équation positionnelle d'une part, l'addition posée colonne par colonne avec quatre retenues entre 0 et 1 d'autre part. Z3 est ensuite interrogé pour un contre-exemple dans chaque sens entre les deux encodages, les retenues étant des variables auxiliaires éliminées existentiellement.

In [8]:
# Exemple guide 4 : SEND + MORE = MONEY avec et sans retenues
# Realisation de @Will-emma, issue de la PR #28098 (conservée telle quelle)
S, E, N, D, M, O, R, Y = Ints('S E N D M O R Y')
lettres = [S, E, N, D, M, O, R, Y]
c1, c2, c3, c4 = Ints('c1 c2 c3 c4')          # les 4 retenues
retenues = [c1, c2, c3, c4]

# Regles communes : chiffres 0..9, tous differents, tetes non nulles
base = And([And(x >= 0, x <= 9) for x in lettres] + [Distinct(lettres), S > 0, M > 0])

# Encodage 1 : une seule equation positionnelle (sans retenue)
positionnel = (1000*S + 100*E + 10*N + D) + (1000*M + 100*O + 10*R + E) == \
              (10000*M + 1000*O + 100*N + 10*E + Y)

# Encodage 2 : l'addition posee colonne par colonne, retenues entre 0 et 1
colonnes = And([And(c >= 0, c <= 1) for c in retenues] + [
    D + E      == Y + 10*c1,   # unites
    N + R + c1 == E + 10*c2,   # dizaines
    E + O + c2 == N + 10*c3,   # centaines
    S + M + c3 == O + 10*c4,   # milliers
    c4 == M,                   # dizaines de milliers
])

# Les retenues sont des variables auxiliaires : "il existe des retenues qui marchent"
colonnes_existe = Exists(retenues, colonnes)

# Sens 1 : solution des colonnes qui ne verifie pas l'equation positionnelle ?
s1 = Solver()
s1.add(base, colonnes, Not(positionnel))
print("Colonnes => positionnel ? contre-exemple :", s1.check())

# Sens 2 : solution positionnelle pour laquelle aucune retenue ne marche ?
s2 = Solver()
s2.add(base, positionnel, Not(colonnes_existe))
print("Positionnel => colonnes ? contre-exemple :", s2.check())

# unsat dans les deux sens = aucun contre-exemple : les deux encodages sont equivalents
# (sat aurait donne un contre-exemple, unknown = Z3 n'a pas pu conclure)

Colonnes => positionnel ? contre-exemple : unsat
Positionnel => colonnes ? contre-exemple : unsat


### Exemple guidé 5 — Changer de base

Réalisation de @Will-emma, issue de la PR #28098.

Pour chaque base entière fixée de 4 à 10 incluse, `AB + BA = CC` est modélisé avec les poids positionnels propres à la base, trois chiffres distincts non nuls ; chaque modèle est vérifié par substitution. Z3 répond `sat` dans toutes les bases avec le même témoin `A=2, B=1, C=3` : l'équation se réduit à `A + B = C`, indépendamment de la base.

In [9]:
# Exemple guide 5 : AB + BA = CC dans les bases 4 a 10
# Realisation de @Will-emma, issue de la PR #28098 (conservée telle quelle)
A, B, C = Ints('A B C')

for b in range(4, 11):
    s = Solver()
    s.add([And(x >= 1, x <= b - 1) for x in (A, B, C)])   # chiffres non nuls de la base
    s.add(Distinct(A, B, C))
    s.add((A*b + B) + (B*b + A) == C*b + C)               # AB + BA = CC en base b

    statut = s.check()
    if statut == sat:
        m = s.model()
        a, bb, c = m[A].as_long(), m[B].as_long(), m[C].as_long()
        verif = (a*b + bb) + (bb*b + a) == c*b + c         # verification par substitution
        print(f"base {b:2d} : sat     A={a} B={bb} C={c}  verifie = {verif}")
    elif statut == unsat:
        print(f"base {b:2d} : unsat   aucune solution")
    else:
        print(f"base {b:2d} : unknown Z3 n'a pas conclu")

base  4 : sat     A=2 B=1 C=3  verifie = True
base  5 : sat     A=2 B=1 C=3  verifie = True
base  6 : sat     A=2 B=1 C=3  verifie = True
base  7 : sat     A=2 B=1 C=3  verifie = True
base  8 : sat     A=2 B=1 C=3  verifie = True
base  9 : sat     A=2 B=1 C=3  verifie = True
base 10 : sat     A=2 B=1 C=3  verifie = True


### Exemple guidé 6 — Expliquer une contradiction

Réalisation de @Will-emma, issue de la PR #28098.

Reprise de `ABC * D = DBC` de l'exemple guidé 3 : les groupes de contraintes reçoivent des étiquettes via `assert_and_track`, le noyau insatisfiable est extrait puis vérifié, et chaque groupe est retiré un par un pour distinguer les groupes nécessaires des groupes inutiles — un noyau n'est pas nécessairement minimal.

In [10]:
# Exemple guide 6 : expliquer pourquoi ABC * D = DBC est impossible
# Realisation de @Will-emma, issue de la PR #28098 (conservée telle quelle)
A, B, C, D = Ints('A B C D')
lettres = [A, B, C, D]

# Les 4 groupes de contraintes, chacun avec une etiquette
groupes = {
    "domaine":   And([And(x >= 0, x <= 9) for x in lettres]),
    "distinct":  Distinct(lettres),
    "tetes":     And(A > 0, D > 0),
    "equation":  (100*A + 10*B + C) * D == 100*D + 10*B + C,
}

s = Solver()
for nom, contrainte in groupes.items():
    s.assert_and_track(contrainte, Bool(nom))

statut = s.check()
print("Statut :", statut)

if statut == unsat:
    noyau = [str(e) for e in s.unsat_core()]
    print("Noyau insatisfiable :", noyau)

    # Verification : les seuls groupes du noyau suffisent-ils a la contradiction ?
    s2 = Solver()
    s2.add([groupes[nom] for nom in noyau])
    print("Groupes du noyau seuls :", s2.check())

    # Le noyau est-il minimal ? On retire chaque groupe un par un et on re-teste
    print("\nEn retirant un groupe du noyau :")
    for retire in noyau:
        s3 = Solver()
        s3.add([groupes[nom] for nom in noyau if nom != retire])
        r = s3.check()
        verdict = "toujours impossible -> ce groupe est inutile" if r == unsat else "devient possible -> ce groupe est necessaire"
        print(f"  sans {retire:9s} : {r}  ({verdict})")
elif statut == unknown:
    print("unknown : Z3 n'a pas conclu, on ne peut rien dire")

# Le noyau n'est pas forcement minimal : Z3 renvoie un ensemble suffisant pour la
# contradiction, pas le plus petit possible. Le test ci-dessus le montre.
# unsat = contradiction prouvee ; unknown = aucune conclusion.

Statut : unsat
Noyau insatisfiable : ['domaine', 'tetes', 'distinct', 'equation']
Groupes du noyau seuls : unsat

En retirant un groupe du noyau :


  sans domaine   : sat  (devient possible -> ce groupe est necessaire)


  sans tetes     : unsat  (toujours impossible -> ce groupe est inutile)
  sans distinct  : sat  (devient possible -> ce groupe est necessaire)
  sans equation  : sat  (devient possible -> ce groupe est necessaire)


### Exemple guidé 7 — Multiplication par constante

Réalisation de @Will-emma, issue de la PR #28098.

`ABCD * 4 = DCBA` est modélisé comme une équation positionnelle où le multiplicateur est la constante 4 ; après résolution, la négation du modèle est ajoutée puis re-testée, ce qui établit l'unicité.

In [11]:
# Exemple guide 7 : ABCD * 4 = DCBA (multiplication)
# Realisation de @Will-emma, issue de la PR #28098 (conservée telle quelle)
A, B, C, D = Ints('A B C D')
lettres = [A, B, C, D]

s = Solver()
s.add([And(x >= 0, x <= 9) for x in lettres])   # 1. chiffres 0..9
s.add(Distinct(lettres))                         # 2. tous differents
s.add(A > 0, D > 0)                              #    tetes non nulles
ABCD = 1000*A + 100*B + 10*C + D
DCBA = 1000*D + 100*C + 10*B + A
s.add(ABCD * 4 == DCBA)                          # 3. equation

if s.check() == sat:
    m = s.model()
    result = {str(x): m[x].as_long() for x in lettres}
    print(m.evaluate(ABCD), "* 4 =", m.evaluate(DCBA))

    # 4. Unicite : on interdit cette solution et on re-teste
    s.add(Or([x != m[x] for x in lettres]))
    if s.check() == unsat:
        print("unsat : la solution est unique")
    else:
        print("autre solution :", s.model())
else:
    result = None
    print("Pas de solution")

2178 * 4 = 8712
unsat : la solution est unique


## Exercices à compléter

### Exercice 1 — Énumérer les solutions d'une addition

Modélisez `TWO + TWO = FOUR` (six lettres distinctes, têtes `T` et `F` non nulles) et énumérez **toutes** les solutions avec la technique de l'exemple guidé 2 : après chaque modèle, interdisez-le par une disjonction et relancez. Pour chaque solution trouvée, vérifiez-la par substitution **dans les deux encodages** de l'exemple guidé 4 : l'équation positionnelle d'une part, l'addition posée colonne par colonne avec retenues d'autre part. Le statut terminal distingue l'énumération exhaustive (`unsat`) d'un arrêt sans preuve (`unknown`).

In [12]:
# Exercice 1 : enumerer les solutions de TWO + TWO = FOUR
# TODO etudiant :
# 1. T, W, O, F, U, R = Ints('T W O F U R'), domaine 0..9
# 2. Distinct sur les 6 lettres, tetes T > 0 et F > 0
# 3. Equation positionnelle : (100*T + 10*W + O) * 2 == 1000*F + 100*O + 10*U + R
# 4. Boucle d'enumeration : bloquer chaque modele par Or([x != m[x] pour chaque lettre])
# 5. Pour chaque solution : verification par substitution dans l'equation positionnelle
#    PUIS dans l'addition posee en colonnes avec retenues (r1, r2, r3 entre 0 et 1)
nb_solutions = None  # TODO etudiant
print("Exercice 1 a completer : enumerez les solutions de TWO + TWO = FOUR")

Exercice 1 a completer : enumerez les solutions de TWO + TWO = FOUR


### Exercice 2 — Bases paires, bases impaires

Pour chaque base entière fixée de 4 à 12 incluse, modélisez `AB + BC = CA` : trois chiffres distincts, non nuls, avec les poids positionnels propres à la base. Affichez le statut pour chaque base et, lorsqu'il existe, un modèle vérifié par substitution. Le comportement n'est pas uniforme : observez quelles bases acceptent une solution, puis expliquez le pattern — l'identité `b * (A + B - C) = A - B - C` est un bon point de départ. Distinguez absence de solution (`unsat`) et réponse `unknown`.

In [13]:
# Exercice 2 : AB + BC = CA dans les bases 4 a 12
# TODO etudiant :
# 1. A, B, C = Ints('A B C')
# 2. Pour chaque base b de 4 a 12 : chiffres non nuls de la base, Distinct(A, B, C),
#    equation (A*b + B) + (B*b + C) == C*b + A
# 3. Afficher le statut par base et verifier chaque modele trouve par substitution
nb_solutions_par_base = None  # TODO etudiant
print("Exercice 2 a completer : cryptarithme en bases variables")

Exercice 2 a completer : cryptarithme en bases variables


### Exercice 3 — Minimiser un noyau insatisfiable

L'exemple guidé 6 extrait un noyau, mais celui-ci n'est pas nécessairement **minimal** : il peut contenir une contrainte que l'on peut retirer sans rendre le problème satisfiable. Reprenez le problème `ABC * D = DBC` et ses quatre groupes suivis (`domaines`, `distinct`, `tetes`, `equation`), puis ajoutez un groupe `redondant` contenant les deux contraintes `A <= 9` et `D >= 2`. Ces bornes sont déjà impliquées par les domaines et les têtes.

**Étapes** :
1. Étiquetez chaque contrainte par un nom unique combinant son groupe et sa position, avec `assert_and_track`, puis affichez le **noyau brut** de Z3. Ne présupposez pas quels groupes il contiendra.
2. Écrivez une fonction qui part de ce noyau et essaie de retirer chaque groupe : si le sous-ensemble restant est encore `unsat`, supprimez ce groupe ; sinon, conservez-le. Recommencez jusqu'à ce qu'aucun retrait ne conserve l'insatisfiabilité.
3. Vérifiez le noyau minimal obtenu dans un nouveau `Solver`, comparez-le au noyau brut et nommez les groupes retirés. Testez chaque sous-groupe élémentaire de ce noyau minimal pour confirmer la minimalité.

**Indice** : un noyau minimal n'est pas nécessairement de cardinal le plus petit parmi tous les noyaux possibles. L'algorithme demandé garantit qu'aucun groupe restant n'est inutile, pas l'optimum global.


In [14]:
# Exercice 3 : reduire un noyau insatisfiable a un noyau minimal
# TODO etudiant :
# 1. Recreer les groupes de l'exemple 6 et ajouter redondant : [A <= 9, D >= 2]
# 2. assert_and_track avec un nom unique par contrainte ; relever le noyau brut
# 3. Pour chaque groupe du noyau : tester le reste ; si unsat, retirer ce groupe
# 4. Recommencer jusqu'a un point fixe, puis verifier et comparer les deux noyaux
resultat = None  # TODO etudiant
print("Exercice 3 a completer : minimisation d'un noyau insatisfiable")


Exercice 3 a completer : minimisation d'un noyau insatisfiable


### Exercice 4 — Faire varier le multiplicateur

Dans l'exemple guidé 7, le multiplicateur est la constante `4`. On cherche maintenant **tous les couples `(N, k)`** tels que `N * k` soit le renversé de `N`, d'abord à quatre chiffres puis à cinq, avec `k` entier de `2` à `9`. Les chiffres de `N` ne sont pas imposés distincts ; sa tête et celle du résultat sont non nulles.

**Étapes** :
1. Pour chaque longueur, déclarez les chiffres `d[0] ... d[n-1]`, construisez `N = Sum(d[i] * 10**(n-1-i))` et `R = Sum(d[i] * 10**i)`.
2. Déclarez `k = Int('k')` **en dehors** de la boucle sur les valeurs `2..9` : `N * k == R` contient alors un produit de deux variables, donc une contrainte **non linéaire**.
3. Ajoutez domaines et têtes non nulles, puis énumérez tous les modèles par blocage du couple `(N, k)` ; vérifiez chaque solution par une multiplication entière ordinaire.
4. Comparez avec la variante où `k` est fixé à chaque itération. Observez les temps ou le statut du solveur sans en déduire une loi générale à partir d'une seule exécution.

**Question de lecture** : pourquoi la forme à multiplicateur variable sort-elle de l'arithmétique linéaire, alors que l'exemple guidé 7 y reste ?


In [15]:
# Exercice 4 : enumerer N * k = renversement(N), k variable de 2 a 9
# TODO etudiant :
# 1. Pour n = 4 puis 5 : d = Ints(...), N = somme positionnelle, R = renversement
# 2. Domaine 0..9, tetes non nulles, k = Int('k') avec 2 <= k <= 9
# 3. Ajouter N * k == R (non lineaire), enumerer tous les couples par blocage
# 4. Verifier les produits en entiers Python, puis comparer avec k fixe
result = None  # TODO etudiant
print("Exercice 4 a completer : enumeration avec multiplicateur variable")


Exercice 4 a completer : enumeration avec multiplicateur variable


## Conclusion

Les cryptarithmes illustrent le paradigme déclaratif : on exprime les domaines, les contraintes de distinction et l'équation plutôt que l'algorithme de recherche. Les exemples montrent la recherche d'un témoin, l'énumération des modèles et l'insatisfiabilité d'une multiplication ; les exemples guidés, issus de rendus d'étudiants, prolongent ces méthodes avec des retenues explicites, des bases variables, un noyau insatisfiable et une multiplication par constante. Les exercices suivent le même mouvement : énumération exhaustive d'une addition, bases paires contre bases impaires, minimisation par suppression d'un noyau et renversement avec multiplicateur variable, qui introduit une contrainte non linéaire.

Le pont avec la série sœur Z3.Linq (C#) se fait via les exemples portés entre Microsoft.Z3 et pyz3.

***

> Carnet issu de l'EPIC #1206.
